In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd

from app.susceptibility_utils import classify_slope_susceptibility

print("Project root:", project_root)
print("Susceptibility utility imported successfully")

In [ ]:
terrain_path = "../data/processed/terrain_features.parquet"

terrain_features = pd.read_parquet(terrain_path)

print("Rows:", len(terrain_features))
print("Columns:", list(terrain_features.columns))
print(terrain_features.head())

In [ ]:
terrain_features["gsi_susceptibility_class"] = terrain_features[
    "slope_mean"
].apply(classify_slope_susceptibility)

print(
    terrain_features[
        [
            "grid_id",
            "slope_mean",
            "gsi_susceptibility_class",
        ]
    ].head(20)
)

In [ ]:
class_distribution = (
    terrain_features["gsi_susceptibility_class"]
    .value_counts(dropna=False)
)

print(class_distribution)

In [ ]:
missing_susceptibility = terrain_features[
    terrain_features["gsi_susceptibility_class"].isna()
]

print("Missing susceptibility cells:", len(missing_susceptibility))
print(
    "Missing susceptibility with missing slope_mean:",
    missing_susceptibility["slope_mean"].isna().sum()
)
print(
    "Missing susceptibility with missing elevation_mean:",
    missing_susceptibility["elevation_mean"].isna().sum()
)

In [ ]:
terrain_features["is_proxy"] = True

print(
    "PROXY NOTICE: GSI NLSM data was not downloadable from the "
    "accessible GSI portals checked on 2026-08-28. "
    "Susceptibility classes are derived from slope_mean using "
    "TEAM-ASSIGNED cutoffs (8 / 15 / 20 degrees) calibrated to this "
    "district's own measured slope distribution -- NOT values from a "
    "published study. See app/susceptibility_utils.py for the "
    "derivation and docs/model_training_log.md section 7.2. "
    "93 grid cells have no DEM coverage and therefore have "
    "null susceptibility class values."
)

print(
    terrain_features[
        [
            "grid_id",
            "gsi_susceptibility_class",
            "is_proxy",
        ]
    ].head(20)
)

In [ ]:
susceptibility_features = terrain_features[
    [
        "grid_id",
        "gsi_susceptibility_class",
        "is_proxy",
    ]
].copy()

print("Rows:", len(susceptibility_features))
print("Columns:", list(susceptibility_features.columns))
print(susceptibility_features.head())

In [ ]:
output_path = "../data/processed/susceptibility_features.parquet"

susceptibility_features.to_parquet(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(susceptibility_features))
print("Columns:", list(susceptibility_features.columns))

In [ ]:
print("Total grid cells:", len(susceptibility_features))
print(
    "Assigned susceptibility classes:",
    susceptibility_features["gsi_susceptibility_class"].notna().sum()
)
print(
    "Null susceptibility classes:",
    susceptibility_features["gsi_susceptibility_class"].isna().sum()
)

print("\nClass distribution:")
print(
    susceptibility_features["gsi_susceptibility_class"]
    .value_counts(dropna=False)
)

print(
    "\nAll rows marked as proxy:",
    susceptibility_features["is_proxy"].all()
)